This document is inteded to showcase usage and allow the IBL team to acess example data until we decide about uploading the full dataset. The idea is to iterate and add improvemenets to the structure, suggest visualization and data access patterns that might be desirable for the final documentation.

To use it, you will need to run this notebook in an environment that has both pynwb and remfile installed plus any additional dependencies that we use for analysis such as matplotib, numpy, pandas, etc.

# Locate Examples

In [2]:
import h5py
import remfile
from pynwb import NWBHDF5IO
from dandi.dandiapi import DandiAPIClient

# Connect to DANDI and get the dandiset
dandiset_id = "000409"
client = DandiAPIClient()
dandiset = client.get_dandiset(dandiset_id, "draft")

# =============================================================================
# Session EIDs for NEW format files (desc-raw / desc-processed)
# =============================================================================

# Complete session with 2 probes - Session 1 (NYU-39, 2021-05-10, angelakilab)
# - Full data: all videos, pose estimation, spike sorting for both probes
TWO_PROBE_SESSION_EID_1 = "6ed57216-498d-48a6-b48b-a243a34710ea"

# Complete session with 2 probes - Session 2 (NYU-39, 2021-05-11, angelakilab)
# - Full data: all videos, pose estimation, spike sorting for both probes
TWO_PROBE_SESSION_EID_2 = "35ed605c-1a1a-47b1-86ff-2b56144f55af"

# Complete session with 1 probe (NYU-46, 2021-06-25, angelakilab)  
# - Full data: all videos, pose estimation, spike sorting for probe01
ONE_PROBE_SESSION_EID = "64e3fb86-928c-4079-865c-b364205b502e"

# Choose which session to use
session_eid = TWO_PROBE_SESSION_EID_1  # Change to TWO_PROBE_SESSION_EID_2 or ONE_PROBE_SESSION_EID

# =============================================================================
# Fetch assets by EID
# =============================================================================

# First, filter assets by EID
session_assets = [asset for asset in dandiset.get_assets() if session_eid in asset.path]

# Then, extract raw and processed files
raw_asset = next((asset for asset in session_assets if "desc-raw" in asset.path), None)
processed_asset = next((asset for asset in session_assets if "desc-processed" in asset.path), None)

print(f"Session EID: {session_eid}")
print(f"\nRaw file:       {raw_asset.path if raw_asset else 'Not found'}")
print(f"Processed file: {processed_asset.path if processed_asset else 'Not found'}")

Session EID: 6ed57216-498d-48a6-b48b-a243a34710ea

Raw file:       sub-NYU-39/sub-NYU-39_ses-6ed57216-498d-48a6-b48b-a243a34710ea_desc-raw_ecephys.nwb
Processed file: sub-NYU-39/sub-NYU-39_ses-6ed57216-498d-48a6-b48b-a243a34710ea_desc-processed_behavior+ecephys.nwb


# Raw NWBFile 

In [3]:
s3_url = raw_asset.get_content_url(follow_redirects=1, strip_query=False)
file_system = remfile.File(s3_url)
file = h5py.File(file_system, mode="r")

io = NWBHDF5IO(file=file)
nwbfile_raw = io.read()

nwbfile_raw

Data type,int16
Shape,"(109132602, 384)"
Array size,78.06 GiB
Chunk shape,"(13020, 384)"
Compression,gzip
Compression opts,4
Uncompressed size (bytes),83813838336
Compressed size (bytes),41280801811
Compression ratio,2.0303345540557385
Data type,float64
Shape,"(109132602,)"


In [4]:
probe_0_ap_series = nwbfile_raw.acquisition["ElectricalSeriesProbe00AP"]
probe_1_ap_series = nwbfile_raw.acquisition["ElectricalSeriesProbe01AP"]

print(f"Probe 0 AP data shape: {probe_0_ap_series.data.shape}")
print(f"Probe 1 AP data shape: {probe_1_ap_series.data.shape}")

probe_0_ap_series.electrodes.to_dataframe()

Probe 0 AP data shape: (109132602, 384)
Probe 1 AP data shape: (109132620, 384)


,location,group,electrode_name,contact_shapes,beryl_location,cosmos_location,channel_name,rel_x,rel_y,contact_ids,shank_ids,adc_group,adc_sample_order,group_name,x,y,z,imp,filtering
id,,,,,,,,,,,,,,,,,,,
384,PO,NeuropixelsProbe00 pynwb.ecephys.ElectrodeGrou...,e0,square,PO,TH,"AP0,LF0",16.0,0.0,e0,,0,0,NeuropixelsProbe00,7657.0,4310.0,4441.0,NaN,
385,PO,NeuropixelsProbe00 pynwb.ecephys.ElectrodeGrou...,e1,square,PO,TH,"AP1,LF1",48.0,0.0,e1,,1,0,NeuropixelsProbe00,7657.0,4310.0,4441.0,NaN,
386,PO,NeuropixelsProbe00 pynwb.ecephys.ElectrodeGrou...,e2,square,PO,TH,"AP2,LF2",0.0,20.0,e2,,0,1,NeuropixelsProbe00,7655.0,4291.0,4436.0,NaN,
387,PO,NeuropixelsProbe00 pynwb.ecephys.ElectrodeGrou...,e3,square,PO,TH,"AP3,LF3",32.0,20.0,e3,,1,1,NeuropixelsProbe00,7655.0,4291.0,4436.0,NaN,
388,PO,NeuropixelsProbe00 pynwb.ecephys.ElectrodeGrou...,e4,square,PO,TH,"AP4,LF4",16.0,40.0,e4,,0,2,NeuropixelsProbe00,7652.0,4272.0,4431.0,NaN,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
763,VISa2/3,NeuropixelsProbe00 pynwb.ecephys.ElectrodeGrou...,e379,square,VISa,Isocortex,"AP379,LF379",32.0,3780.0,e379,,31,9,NeuropixelsProbe00,7275.0,657.0,3478.0,NaN,
764,VISa2/3,NeuropixelsProbe00 pynwb.ecephys.ElectrodeGrou...,e380,square,VISa,Isocortex,"AP380,LF380",16.0,3800.0,e380,,30,10,NeuropixelsProbe00,7275.0,638.0,3471.0,NaN,
765,VISa2/3,NeuropixelsProbe00 pynwb.ecephys.ElectrodeGrou...,e381,square,VISa,Isocortex,"AP381,LF381",48.0,3800.0,e381,,31,10,NeuropixelsProbe00,7275.0,638.0,3471.0,NaN,


# Processed files

In [5]:
s3_url = processed_asset.get_content_url(follow_redirects=1, strip_query=False)
file_system = remfile.File(s3_url)
file = h5py.File(file_system, mode="r")

io = NWBHDF5IO(file=file)
nwbfile_processed = io.read()

nwbfile_processed

root pynwb.file.NWBFile at 0x5104466384
Fields:
  devices: {
    NeuropixelsProbe00 <class 'pynwb.device.Device'>,
    NeuropixelsProbe01 <class 'pynwb.device.Device'>
  }
  electrode_groups: {
    NeuropixelsProbe00 <class 'pynwb.ecephys.ElectrodeGroup'>,
    NeuropixelsProbe01 <class 'pynwb.ecephys.ElectrodeGroup'>
  }
  electrodes: electrodes <class 'pynwb.ecephys.ElectrodesTable'>
  epochs: epochs <class 'pynwb.epoch.TimeIntervals'>
  file_create_date: [datetime.datetime(2025, 12, 9, 10, 0, 44, 357195, tzinfo=tzutc())]
  identifier: eccf9d97-5e31-4d48-88a0-c10507d03c47
  institution: New York University / Center for Neural Science
  intervals: {
    epochs <class 'pynwb.epoch.TimeIntervals'>,
    trials <class 'pynwb.epoch.TimeIntervals'>
  }
  lab: angelakilab
  lab_meta_data: {
    ibl_bwm_metadata <class 'abc.ibl_bwm_metadata'>,
    localization <class 'abc.Localization'>
  }
  processing: {
    behavior_events <class 'pynwb.base.ProcessingModule'>,
    passive <class 'pynwb.base.ProcessingModule'>,
    pose_estimation <class 'pynwb.base.ProcessingModule'>,
    video <class 'pynwb.base.ProcessingModule'>,
    wheel <class 'pynwb.base.ProcessingModule'>
  }
  protocol: _iblrig_tasks_ephysChoiceWorld6.4.2
  session_id: 6ed57216-498d-48a6-b48b-a243a34710ea
  session_start_time: 2021-05-10 14:33:49.023776-04:00
  source_script: Created using NeuroConv v0.9.1
  source_script_file_name: /ebs/IBL-to-nwb/.venv/lib/python3.10/site-packages/neuroconv/basedatainterface.py
  subject: subject abc.IblSubject at 0x5172170768
Fields:
  age__reference: birth
  date_of_birth: 2020-12-08 00:00:00-05:00
  expected_water_ml: 0.856
  remaining_water_ml: 0.856
  sex: M
  species: Mus musculus
  subject_id: NYU-39
  uuid: ccb12ddd-95c4-4fc0-956f-db2f8cdcd26d
  weight: 0.022 kg

  timestamps_reference_time: 2021-05-10 14:33:49.023776-04:00
  trials: trials <class 'pynwb.epoch.TimeIntervals'>
  units: units <class 'pynwb.misc.Units'>

In [10]:
nwbfile_processed.session_start_time

datetime.datetime(2021, 5, 10, 14, 33, 49, 23776, tzinfo=tzoffset(None, -14400))

In [11]:
nwbfile_processed.subject

subject abc.IblSubject at 0x5172170768
Fields:
  age__reference: birth
  date_of_birth: 2020-12-08 00:00:00-05:00
  expected_water_ml: 0.856
  remaining_water_ml: 0.856
  sex: M
  species: Mus musculus
  subject_id: NYU-39
  uuid: ccb12ddd-95c4-4fc0-956f-db2f8cdcd26d
  weight: 0.022 kg

In [12]:
len(nwbfile_processed.trials)

541

In [6]:
units_df = nwbfile_processed.units.to_dataframe()
trials_df = nwbfile_processed.trials.to_dataframe()
electrodes_df = nwbfile_processed.electrodes.to_dataframe()

In [14]:
trials_df.shape

(541, 14)

In [15]:
units_df.shape

(1366, 25)

In [ ]:
units_df.columns

Index(['spike_times', 'electrodes', 'unit_name', 'label',
       'alternative_contamination', 'median_amplitude',
       'presence_ratio_standard_deviation', 'cluster_id', 'presence_ratio',
       'noise_cutoff', 'firing_rate', 'maximum_amplitude',
       'sliding_refractory_period_violation', 'contamination', 'cluster_uuid',
       'maximum_amplitude_channel', 'mean_relative_depth', 'spike_count',
       'drift', 'spike_relative_depths', 'missed_spikes_estimate',
       'minimum_amplitude', 'standard_deviation_amplitude', 'spike_amplitudes',
       'probe_name'],
      dtype='object')

In [7]:
units_df.columns

Index(['spike_times', 'electrodes', 'unit_name', 'label',
       'alternative_contamination', 'median_amplitude',
       'presence_ratio_standard_deviation', 'cluster_id', 'presence_ratio',
       'noise_cutoff', 'firing_rate', 'maximum_amplitude',
       'sliding_refractory_period_violation', 'contamination', 'cluster_uuid',
       'maximum_amplitude_channel', 'mean_relative_depth', 'spike_count',
       'drift', 'spike_relative_depths', 'missed_spikes_estimate',
       'minimum_amplitude', 'standard_deviation_amplitude', 'spike_amplitudes',
       'probe_name'],
      dtype='object')

In [13]:
nwbfile_processed.trials.to_dataframe()

,start_time,stop_time,choice,feedback_type,reward_volume,contrast_left,contrast_right,probability_left,feedback_time,response_time,stim_off_time,stim_on_time,go_cue_time,first_movement_time
id,,,,,,,,,,,,,,
0,49.622837,52.025851,1.0,1.0,1.5,1.000,NaN,0.5,50.456654,50.456541,51.525847,50.242575,50.243408,50.330254
1,52.488151,56.125782,-1.0,1.0,1.5,NaN,0.125,0.5,54.558977,54.558867,55.625737,54.230543,54.231410,54.416254
2,56.503582,58.792414,1.0,1.0,1.5,0.250,NaN,0.5,57.224877,57.224792,58.292337,57.014298,57.015398,57.136254
3,59.156617,61.480618,-1.0,1.0,1.5,NaN,1.000,0.5,59.917808,59.917717,60.980601,59.697496,59.698396,59.722254
4,61.841715,64.309232,1.0,1.0,1.5,0.000,NaN,0.5,62.754024,62.753921,63.809118,62.475552,62.476385,62.355254
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536,2290.803597,2293.949737,1.0,1.0,1.5,1.000,NaN,0.2,2292.383305,2292.383216,2293.449699,2291.516359,2291.517425,2291.679254
537,2294.672243,2303.194311,-1.0,-1.0,0.0,0.000,NaN,0.2,2300.635713,2300.634788,2302.694241,2295.161028,2295.162061,2300.234254
538,2303.967514,2306.449225,-1.0,1.0,1.5,NaN,0.250,0.2,2304.871623,2304.871517,2305.949149,2304.499194,2304.500027,2304.715254


In [14]:
nwbfile_processed.units.to_dataframe().columns

Index(['spike_times', 'electrodes', 'unit_name', 'label',
       'alternative_contamination', 'median_amplitude',
       'presence_ratio_standard_deviation', 'cluster_id', 'presence_ratio',
       'noise_cutoff', 'firing_rate', 'maximum_amplitude',
       'sliding_refractory_period_violation', 'contamination', 'cluster_uuid',
       'maximum_amplitude_channel', 'mean_relative_depth', 'spike_count',
       'drift', 'spike_relative_depths', 'missed_spikes_estimate',
       'minimum_amplitude', 'standard_deviation_amplitude', 'spike_amplitudes',
       'probe_name'],
      dtype='object')

In [15]:
nwbfile_processed.units.to_dataframe()

,spike_times,electrodes,unit_name,label,alternative_contamination,median_amplitude,presence_ratio_standard_deviation,cluster_id,presence_ratio,noise_cutoff,firing_rate,maximum_amplitude,sliding_refractory_period_violation,contamination,cluster_uuid,maximum_amplitude_channel,mean_relative_depth,spike_count,drift,spike_relative_depths,missed_spikes_estimate,minimum_amplitude,standard_deviation_amplitude,spike_amplitudes,probe_name
id,,,,,,,,,,,,,,,,,,,,,,,,,
0,"[3.4898576492439375, 7.69315187588426, 12.2633...",location ...,0,0.666667,0.396912,0.000253,88.168130,0,0.983562,10.514859,9.986225,0.000370,1.0,0.594632,8a3fb319-d1eb-445b-9d7f-c208067543cc,0,20.0,36327.0,5.332562e+05,"[68.94155883789062, 62.34589385986328, 107.208...",0.046194,0.000133,1.558649,"[0.0001328487415365554, 0.00014406364229215, 0...",probe00
1,"[0.06328952941703922, 0.07255610986910907, 0.2...",location ...,1,0.333333,0.605311,0.000094,192.161516,1,0.997260,405.187456,31.913752,0.000225,0.0,1.040543,b1aa8503-6e2a-4e42-be94-0057db48ec2e,0,20.0,116093.0,4.910825e+06,"[129.95050048828125, 200.1836700439453, 221.37...",0.348336,0.000069,1.471201,"[7.433633596786837e-05, 8.218951586841179e-05,...",probe00
2,"[0.13385553955024748, 0.13482219722330513, 0.1...",location ...,2,0.333333,0.386672,0.000089,70.725500,2,0.997260,260.530651,25.103975,0.000216,0.0,0.574396,57a200c5-e79b-4346-9c64-311cc8553610,1,20.0,91321.0,3.504317e+06,"[229.6371612548828, 134.47418212890625, 157.79...",0.500000,0.000079,1.099140,"[8.325040769107711e-05, 9.543088639567879e-05,...",probe00
3,"[1.7363739633405737, 3.574756859356966, 5.1645...",location ...,3,0.666667,0.065618,0.000081,116.761056,3,0.997260,107.066668,33.853983,0.000167,1.0,0.074955,da179590-df96-455d-8f76-71df8d6e572f,5,60.0,123151.0,4.660627e+06,"[234.13909912109375, 159.41329956054688, 200.3...",0.500000,0.000068,1.150589,"[7.179665701876558e-05, 7.948231897074719e-05,...",probe00
4,"[0.013556658789383686, 0.016289966692512203, 0...",location ...,4,0.666667,0.086135,0.000055,36.469342,4,0.997260,346.026570,24.741659,0.000299,1.0,0.099606,7f94d191-a2e7-487f-89f7-4564bb6326d4,6,80.0,90003.0,2.090179e+06,"[87.53541564941406, 151.60496520996094, 116.04...",0.500000,0.000043,1.451231,"[5.939352773409396e-05, 4.546439293945639e-05,...",probe00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1361,"[1178.4987611741014, 1253.7945003884095, 1263....",location ...,1361,0.666667,4.073714,0.000109,5.174570,913,0.306849,-1.111146,0.215245,0.000147,0.0,50.857811,f7f7cfd3-2997-4b40-9543-ecba6a83549e,302,3040.0,783.0,3.424537e+04,"[2982.500732421875, 3063.069091796875, 3055.95...",0.000667,0.000064,1.011393,"[9.95427330642446e-05, 9.396257703342509e-05, ...",probe01
1362,"[130.06496644164685, 166.41813075526932, 174.3...",location ...,1362,0.666667,4.300486,0.000069,3.748988,914,0.309589,3.383385,0.170987,0.000099,0.0,47.012918,21779608-6ca1-4818-8888-a69bb4336784,304,3060.0,622.0,3.033558e+04,"[2988.23779296875, 3040.6787109375, 3060.10107...",0.370366,0.000056,1.111652,"[6.135298547803103e-05, 6.147037975912716e-05,...",probe01
1363,"[1.62327066355882, 1.653970407166465, 1.744236...",location ...,1363,0.000000,4.586888,0.000049,2.581505,915,0.298630,7.375636,0.113808,0.000088,0.0,30.320004,0c2a6c89-96a9-432b-ab6a-2ea746e05ea8,316,3180.0,414.0,1.211940e+04,"[3218.9111328125, 3201.47314453125, 3227.77490...",NaN,0.000041,1.392962,"[4.09699603921748e-05, 4.8645283034576176e-05,...",probe01
